## Credit Card Fraud Detection

This project makes use of a 2013 European financial transaction dataset to build and contrast linear regressive against gradient boosted ML models in their ability to detect future fraud. With an initial exploratory data analysis section, comparison and contrast of 2 linear regression models, setup and training of an XGBoost model, which then had its threshold for categorising fraud analysed and changed. This leads to section 6, where there is a brief look at prevalence of overfitting on the set by comparing the XGBoost model against a variant to justify our model. At the end, there is a SHAP plot which we use as a final point of discussion in the models evaluation.

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sklearn
import xgboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import (classification_report, confusion_matrix, average_precision_score, roc_auc_score)
from sklearn.metrics import precision_recall_curve
import shap


#modules required and dependencies

In [ ]:
#Load the data frame path, read into pandas

path = r"YOUR_PATH_HERE.CSV"
df = pd.read_csv(path)
df

## 2. Exploratory Data Analysis

In [ ]:
#Check out the df

print(df.dtypes)
print(df.shape)
print(df.info())
print(df.isnull().sum().sum())

The dataframe is clean, time axis, class interger fraud or not, an amount column, and 28 feature columns. No value errors or N/a present

Histograms are produced as part of data exploration for time frequency, quantities of transactions, and the anonymised data columns.

In [ ]:


fraud = np.log1p(df[df['Class'] == 1]['Amount'])
legit = np.log1p(df[df['Class'] == 0]['Amount']) #Create classes with log(1+x) of amount in order to compare both in a density histogram

plt.hist(legit, bins = 50, alpha = 0.5 , density = True, label = 'Legit') #Density True in order to compare despite imbalanced set volumes
plt.hist(fraud, bins = 50, alpha = 0.5, density = True, label = 'Fraud')
plt.xlabel('log1p(Amount)')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
fraud = df[df['Class'] == 1]['Time']
legit = df[df['Class'] == 0]['Time'] #Create classes to compare frequency of fraud in a histogram

plt.hist(legit, bins = 200, alpha = 0.5 , density = True, label = 'Legit')
plt.hist(fraud, bins = 200, alpha = 0.5, density = True, label = 'Fraud')
plt.xlabel('Time')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(16, 20))
for i, ax in enumerate(axes.flat): # iterate over each of the anonymised "V" variables
    col = f'V{i+1}'
    ax.hist(df[df['Class']==0][col], bins=50, alpha=0.5, density=True)
    ax.hist(df[df['Class']==1][col], bins=50, alpha=0.5, density=True)
    ax.set_title(col)
plt.tight_layout()

## 3. Logistic Regression Models as a Baseline

In [ ]:
y = df['Class']
X = df.drop(columns = 'Class') # Class split off as predictive value

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y, test_size=0.2) #Stratify to ensure splits contain fraud results due to sparse distribution 

print(X_train.head())
print(X_test.head())
print(y_train.head())
print(y_test.head()) #Confirming data split looks acceptable

In [ ]:
scaler = StandardScaler() #data is scaled to prevent dominant magnitudes overwhelming learning
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test) #Test data scaled according to the scaled training data

model_0 = LogisticRegression(max_iter=1000)
model_0.fit(X_train, y_train) #Instantiate and fit model to training data

y_preds = model_0.predict(X_test)
y_probability = model_0.predict_proba(X_test)[:,1] #Find predictions and probabilities

print(confusion_matrix(y_test, y_preds))
print(classification_report(y_test, y_preds))
print(average_precision_score(y_test, y_probability))
print(roc_auc_score(y_test, y_probability))

#Stats for this model
LogUnBalanced = [f'LogUnBalanced - PRAUC: {average_precision_score(y_test, y_probability)}, ROCAUC: {roc_auc_score(y_test, y_probability)}']
LogUnBalancedClassification = classification_report(y_test, y_preds)

In [ ]:
model_1 = LogisticRegression(max_iter=1000, class_weight='balanced') #Repeat process for Model 1
model_1.fit(X_train, y_train)

y_preds = model_1.predict(X_test)
y_probability = model_1.predict_proba(X_test)[:,1]

print(confusion_matrix(y_test, y_preds))
print(classification_report(y_test, y_preds))
print(average_precision_score(y_test, y_probability))
print(roc_auc_score(y_test, y_probability))


LogBalanced = [f'LogBalanced -   PRAUC: {average_precision_score(y_test, y_probability)}, ROCAUC: {roc_auc_score(y_test, y_probability)}']
LogBalancedClassification = classification_report(y_test, y_preds)

## 4. XGBoost Model

In [ ]:
y = df['Class'] 
X = df.drop(columns = 'Class') #XGboost moodel data is handled differently, the train test split is recreated without scaling applied.

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y, test_size=0.2)

model_2 = xgboost.XGBClassifier(scale_pos_weight = 580,
                                n_estimators = 100,
                                max_depth = 4,
                                learning_rate = 0.1,
                                eval_metric = 'aucpr',
                                random_state = 42) # model instantiated


model_2.fit(X_train, y_train)
y_preds = model_2.predict(X_test)
y_probability = model_2.predict_proba(X_test)[:,1] #model fitted and predictions made

print(confusion_matrix(y_test, y_preds))
print(classification_report(y_test, y_preds))
print(average_precision_score(y_test, y_probability))
print(roc_auc_score(y_test, y_probability))

XGBoostMod = [f'XGBoostModel -   PRAUC: {average_precision_score(y_test, y_probability)}, ROCAUC: {roc_auc_score(y_test, y_probability)}']
XGBoostModClassification = classification_report(y_test, y_preds)

In [ ]:
print(LogUnBalanced)
print(LogBalanced)
print(XGBoostMod)

print(LogUnBalancedClassification)
print(LogBalancedClassification)
print(XGBoostModClassification)

## 5. Threshold Tune

In [ ]:
prcurve = PrecisionRecallDisplay.from_predictions(y_test, y_probability, despine=True) #Plot precision recall curve
plt.savefig('images/Precision-Recall_Curve.png', bbox_inches='tight')

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_probability) #Plot precision recall threshold curve

plt.plot(thresholds, precision[:-1], label = "Precision")
plt.plot(thresholds, recall[:-1], label = "Recall")
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.legend()
plt.savefig('images/Precision-Recall-Threshold_Curve.png', bbox_inches='tight')

In [ ]:
index = np.where(precision[:-1] >= 0.75)[0] #Locate the position for threshold for a 0.75 precision
precbasedthresh = thresholds[index[0]] #Select as threshold
y_preds = (y_probability>=precbasedthresh).astype(int) #Implement the criterion

print(confusion_matrix(y_test, y_preds))
print(classification_report(y_test, y_preds))
print(XGBoostModClassification) #Compare to the 0.5 threshold

## 6. Overfitting check

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y, test_size=0.2)

train_probability = model_2.predict_proba(X_train)[:, 1]
test_probability  = model_2.predict_proba(X_test)[:, 1]

print("Train PRAUC:", average_precision_score(y_train, train_probability))
print("Test PRAUC :", average_precision_score(y_test, test_probability)) #PR-AUC ran on test and train

In [ ]:
y = df['Class']
X = df.drop(columns = 'Class')

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y, test_size=0.2)

model_3 = xgboost.XGBClassifier(scale_pos_weight = 580,
                                n_estimators = 100,
                                max_depth = 3,
                                learning_rate = 0.05,
                                eval_metric = 'aucpr',
                                random_state = 42) #less aggressive model instantiated, with 1 layer less depth and slower learning rate


model_3.fit(X_train, y_train)
y_preds = model_3.predict(X_test)
y_probability = model_3.predict_proba(X_test)[:,1]

print(f' Confusion Matrix: {confusion_matrix(y_test, y_preds)}')
print(classification_report(y_test, y_preds))
print(f'PR-AUC:{average_precision_score(y_test, y_probability)}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_probability)}')

XGBoostMod2 = [f'XGBoostModel -   PRAUC: {average_precision_score(y_test, y_probability)}, ROCAUC: {roc_auc_score(y_test, y_probability)}']
XGBoostModClassification2 = classification_report(y_test, y_preds)

train_probability = model_3.predict_proba(X_train)[:, 1]
test_probability  = model_3.predict_proba(X_test)[:, 1]

print("Train PRAUC:", average_precision_score(y_train, train_probability))
print("Test PRAUC :", average_precision_score(y_test, test_probability))

## 7. SHAP Interpretation

In [ ]:

shapley_vals = shap.TreeExplainer(model_2).shap_values(X_test) #The initial XGBoost model is fed to SHAP tree explainer
shap.summary_plot(shapley_vals, X_test, show = False)
plt.savefig('images/SHAP_Summary_Top_Features.png', bbox_inches='tight')
plt.show()

## 8. Conclusion

### Comparison ###

To briefly outline the format of the summary plot, features identified as being important to categorisation are at the top, descending. Lower values are in blue, higher in red, and their position on the x-axis indicates which direction the point tells the model it is weighted towards. I.e. V14 clusters in the middle, however in general there is a gradient leading to extremes, with low values telling the model fraud is much more likely, and high values suggesting fraud is less likely.

First we will discuss the anonymised features. Earlier, "V4, 11,12, 14, 17" were identified by visual inspection as being correlated to fraud. The model agreed other than with feature 17. If we look at the histogram for V17, we can see that fraud cases did indeed cluster at 0, however genuine cases sat at an average just under 0 but with a wider distribution. The model deemed this to not be a useful feature, or at least didn't prioritise it, with most points having close to 0 SHAP value, but there is 1 feature worth noting, and that is the tail of low values leading up to near a +2 SHAP value. It is worth noting that there are several other features that weight heavily despite not appearing so drastically important according to our histograms, but we will touch more on this in our analysis.

Let us consider amount and time, which are digestible intuitively. Here we see that the histograms are severely limited for analytical purposes on datasets like this, where one category dominates, since both features are middle of the road according to our XGBoost model. Despite fraud appearing slightly more common at high and low values of log1p amount, our model doesn't gain so much from this. This is where the dataset's limitations are apparent, as we manually boosted signal that maybe wasn't there, and binned it, making it look like a major feature visually.

Regarding time, a similar issue appears, where it looks like fraud never sleeps, and so we might assert that there is signal there for the model. Unfortunately, we boosted fraud in the histograms, and so we can't look at the lack of a distribution as proof of disparity from the distribution as a whole. The model didn't find this feature too useful. As time is a continuous parameter in the data, but over the course of 2 days we see peaks and troughs, we would like the model to pick up on the sinusoidal pattern, however this will likely require feature engineering to achieve, since gradient boosting alone won't approximate this well, especially with the rest of the data being part of the model.